In [ ]:
# Replace only these four paths before running the notebook.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    '/kaggle/input/<offline-packages-dataset>/bm25s-0.3.11-py3-none-any.whl'
)
LEGALIR_SOURCE_PATH = Path('/kaggle/input/<legalir-dataset>/train.json')
CORPUS_PATH = Path('/kaggle/input/<legalir-dataset>/selected-contexts')
MODEL_PATH = Path(
    '/kaggle/input/<bge-reranker-dataset>/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
)


In [ ]:
# Validate attached inputs, set offline mode, and install the pinned BM25 wheel.
import os
import subprocess
import sys
from pathlib import Path

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

if not BM25_WHEEL_PATH.is_file():
    raise FileNotFoundError(f'Attach the offline BM25 wheel at: {BM25_WHEEL_PATH}')
if not LEGALIR_SOURCE_PATH.is_file():
    raise FileNotFoundError(f'Attach the LegalIR source JSON at: {LEGALIR_SOURCE_PATH}')
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f'Attach the LegalIR corpus at: {CORPUS_PATH}')
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f'Attach the complete model snapshot directory at: {MODEL_PATH}')

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
# Standalone LegalIR DEV retrieval and reranking implementation.
import json
import re
from collections import Counter
from collections.abc import Callable
from hashlib import sha256
from math import isfinite
from pathlib import Path
from statistics import median
from time import perf_counter

import bm25s
import numpy as np
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer



def _read_json(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"JSON file does not exist: {path}")

    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_legal_ir(path: str | Path) -> dict:
    """Load a LegalIR JSON object without changing its contents."""

    data = _read_json(Path(path))
    if not isinstance(data, dict) or not all(
        isinstance(sample, dict) for sample in data.values()
    ):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    return data


def _corpus_documents(data, source: str) -> list[dict]:
    if isinstance(data, dict):
        documents = [data]
    elif isinstance(data, list):
        documents = data
    else:
        raise ValueError(f"{source}: expected a document or list of documents")

    if not all(isinstance(document, dict) for document in documents):
        raise ValueError(f"{source}: every corpus document must be an object")
    return documents


def load_corpus(path: str | Path) -> list[dict]:
    """Load corpus documents from a JSON file or directory."""

    input_path = Path(path)
    if input_path.is_dir():
        json_paths = sorted(
            candidate
            for candidate in input_path.rglob("*")
            if candidate.is_file() and candidate.suffix.lower() == ".json"
        )
        if not json_paths:
            raise ValueError(f"{input_path}: directory contains no JSON files")

        documents = []
        for json_path in json_paths:
            documents.extend(_corpus_documents(_read_json(json_path), str(json_path)))
        return documents

    if not input_path.is_file():
        raise FileNotFoundError(f"Input path does not exist: {input_path}")
    if input_path.suffix.lower() == ".json":
        return _corpus_documents(_read_json(input_path), str(input_path))
    raise ValueError(f"{input_path}: expected a JSON file or directory")


def make_legalir_split(samples: dict) -> dict[str, list[str]]:
    """Assign exact-question groups to the fixed LegalIR split-v1 buckets."""

    split_ids = {"train": [], "dev": [], "holdout": []}
    for sample_id, sample in samples.items():
        canonical_id = str(sample_id)
        question = sample.get("question")
        group_key = (
            question
            if isinstance(question, str)
            else f"\0fallback-sample-id:{canonical_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        split = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        split_ids[split].append(canonical_id)

    for ids in split_ids.values():
        ids.sort()
    return split_ids


def select_samples(samples: dict, sample_ids: list[str]) -> dict:
    """Select a manifest-defined subset while preserving manifest ID order."""

    canonical_samples = {str(sample_id): sample for sample_id, sample in samples.items()}
    missing = [sample_id for sample_id in sample_ids if sample_id not in canonical_samples]
    if missing:
        raise KeyError(f"sample IDs are absent from source data: {missing[:5]}")
    return {sample_id: canonical_samples[sample_id] for sample_id in sample_ids}


def _validate_window_parameters(chunk_size: int, overlap: int) -> None:
    if not isinstance(chunk_size, int) or isinstance(chunk_size, bool):
        raise TypeError("chunk_size must be an integer")
    if not isinstance(overlap, int) or isinstance(overlap, bool):
        raise TypeError("overlap must be an integer")
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")


def chunk_document(
    document: dict, chunk_size: int = 2_000, overlap: int = 200
) -> list[dict]:
    """Split one document into source-preserving character windows.

    Empty passages produce no chunks. Every chunk's half-open offsets refer
    directly to the unmodified ``document["passage"]`` string.
    """

    _validate_window_parameters(chunk_size, overlap)

    document_id = str(document["id"])
    text = document["passage"]
    if not isinstance(text, str):
        raise TypeError(f"document {document_id!r}: passage must be a string")
    if not text:
        return []

    step = chunk_size - overlap
    chunks = []
    for chunk_index, char_start in enumerate(range(0, len(text), step)):
        char_end = min(char_start + chunk_size, len(text))
        chunks.append(
            {
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "chunk_index": chunk_index,
                "text": text[char_start:char_end],
                "char_start": char_start,
                "char_end": char_end,
            }
        )
        if char_end == len(text):
            break

    return chunks


def chunk_corpus(
    documents: list[dict], chunk_size: int = 2_000, overlap: int = 200
) -> list[dict]:
    """Return fixed-window chunks for corpus documents in input order."""

    _validate_window_parameters(chunk_size, overlap)

    chunks = []
    seen_document_ids = set()
    for document in documents:
        document_id = str(document["id"])
        if document_id in seen_document_ids:
            raise ValueError(
                f"duplicate document ID after canonicalization: {document_id}"
            )
        seen_document_ids.add(document_id)
        chunks.extend(chunk_document(document, chunk_size=chunk_size, overlap=overlap))
    return chunks


AGGREGATION_METHODS = ("max", "mean_top_2", "mean_top_3", "sum_top_2")


def aggregate_documents(chunk_hits: list[dict], method: str = "max") -> list[dict]:
    """Aggregate one query's chunk hits into a deterministic document ranking.

    Ties use the best (lowest) supporting chunk rank, then the canonical string
    document ID. Every unique document in ``chunk_hits`` appears exactly once.
    """

    if method not in AGGREGATION_METHODS:
        raise ValueError(
            f"unknown aggregation method {method!r}; expected one of "
            f"{AGGREGATION_METHODS}"
        )

    grouped: dict[str, dict] = {}
    for hit in chunk_hits:
        document_id = str(hit["document_id"])
        chunk_rank = hit["chunk_rank"]
        if not isinstance(chunk_rank, int) or isinstance(chunk_rank, bool):
            raise TypeError("chunk_rank must be an integer")
        if chunk_rank <= 0:
            raise ValueError("chunk_rank must be greater than zero")

        score = float(hit["score"])
        if not isfinite(score):
            raise ValueError("chunk score must be finite")

        document = grouped.setdefault(
            document_id,
            {"scores": [], "best_chunk_rank": chunk_rank},
        )
        document["scores"].append(score)
        document["best_chunk_rank"] = min(document["best_chunk_rank"], chunk_rank)

    ranked_documents = []
    for document_id, document in grouped.items():
        scores = sorted(document["scores"], reverse=True)
        if method == "max":
            aggregate_score = scores[0]
        elif method == "mean_top_2":
            selected_scores = scores[:2]
            aggregate_score = sum(selected_scores) / len(selected_scores)
        elif method == "mean_top_3":
            selected_scores = scores[:3]
            aggregate_score = sum(selected_scores) / len(selected_scores)
        else:
            aggregate_score = sum(scores[:2])

        ranked_documents.append(
            {
                "document_id": document_id,
                "score": aggregate_score,
                "best_chunk_rank": document["best_chunk_rank"],
                "supporting_chunk_count": len(scores),
            }
        )

    ranked_documents.sort(
        key=lambda document: (
            -document["score"],
            document["best_chunk_rank"],
            document["document_id"],
        )
    )
    return ranked_documents


TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def lexical_tokenize(text: str) -> list[str]:
    """Lowercase and split text into Unicode letter/digit/underscore tokens."""

    if not isinstance(text, str):
        raise TypeError("text must be a string")
    return TOKEN_PATTERN.findall(text.lower())


def _sparse_index_size_bytes(retriever: bm25s.BM25) -> int:
    return sum(
        value.nbytes
        for value in retriever.scores.values()
        if isinstance(value, np.ndarray)
    )


def build_bm25(
    chunks: list[dict], k1: float = 1.5, b: float = 0.75
) -> dict:
    """Build a bm25s Lucene-style sparse index and retain compact chunk metadata."""

    if not chunks:
        raise ValueError("chunks must not be empty")

    started = perf_counter()
    tokenized_chunks = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True,
        token_pattern=r"(?u)\w+",
        stopwords=[],
        stemmer=None,
        return_ids=True,
        show_progress=False,
    )
    retriever = bm25s.BM25(k1=k1, b=b, method="lucene")
    retriever.index(tokenized_chunks, show_progress=False)

    return {
        "retriever": retriever,
        "chunk_metadata": [
            (str(chunk["chunk_id"]), str(chunk["document_id"])) for chunk in chunks
        ],
        "number_of_chunks": len(chunks),
        "k1": k1,
        "b": b,
        "method": "lucene",
        "library": "bm25s",
        "library_version": bm25s.__version__,
        "build_seconds": perf_counter() - started,
        "sparse_index_size_bytes": _sparse_index_size_bytes(retriever),
    }


def make_legalir_predictions(
    rankings: dict[str, list[str]], top_k: int = 5
) -> dict[str, dict[str, list[str]]]:
    """Build the prediction object consumed by the bundled LegalIR scorer."""

    if not isinstance(top_k, int) or isinstance(top_k, bool):
        raise TypeError("top_k must be an integer")
    if not 0 < top_k <= 5:
        raise ValueError("top_k must be between one and five")

    predictions = {}
    for sample_id, ranked_document_ids in rankings.items():
        top_ids = [str(document_id) for document_id in ranked_document_ids[:top_k]]
        if len(top_ids) != len(set(top_ids)):
            raise ValueError(
                f"sample {sample_id!r}: top-{top_k} ranking contains duplicate IDs"
            )
        predictions[sample_id] = {"answer": top_ids}
    return predictions


def _percentile(values: list[int], percent: int) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * percent / 100
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return float(ordered[lower] + (ordered[upper] - ordered[lower]) * fraction)


def evaluate_retrieval(
    samples: dict,
    rankings: dict[str, list[str]],
    candidate_depths: tuple[int, ...] = (10, 20, 50, 100, 200),
) -> dict:
    """Evaluate candidate recall and official-style top-5 set-overlap scores."""

    recall_values = {depth: [] for depth in candidate_depths}
    reciprocal_ranks = []
    top_5_precision = []
    top_5_recall = []
    zero_recall_at_100_ids = []
    full_recall_at_100 = 0
    gold_only_after_rank_5 = 0

    for sample_id, sample in samples.items():
        gold_values = sample.get("answer")
        if not isinstance(gold_values, list) or not gold_values:
            raise ValueError(f"sample {sample_id!r}: expected a non-empty answer list")

        gold = {str(document_id) for document_id in gold_values}
        ranked = [str(document_id) for document_id in rankings.get(str(sample_id), [])]
        if len(ranked) != len(set(ranked)):
            raise ValueError(f"sample {sample_id!r}: ranking contains duplicate IDs")

        for depth in candidate_depths:
            recall_values[depth].append(len(gold.intersection(ranked[:depth])) / len(gold))

        first_gold_rank = next(
            (rank for rank, document_id in enumerate(ranked, start=1) if document_id in gold),
            None,
        )
        reciprocal_ranks.append(0.0 if first_gold_rank is None else 1 / first_gold_rank)
        if first_gold_rank is not None and first_gold_rank > 5:
            gold_only_after_rank_5 += 1

        top_5 = ranked[:5]
        overlap = len(gold.intersection(top_5))
        top_5_precision.append(overlap / len(top_5) if 0 < len(top_5) <= 5 else 0.0)
        top_5_recall.append(overlap / len(gold) if 0 < len(top_5) <= 5 else 0.0)

        recall_at_100 = len(gold.intersection(ranked[:100])) / len(gold)
        if recall_at_100 == 0:
            zero_recall_at_100_ids.append(str(sample_id))
        if recall_at_100 == 1:
            full_recall_at_100 += 1

    number_of_queries = len(samples)
    if number_of_queries == 0:
        raise ValueError("samples must not be empty")

    candidate_recall = {}
    for depth, values in recall_values.items():
        candidate_recall[depth] = {
            "mean": sum(values) / number_of_queries,
            "zero_recall_rate": sum(value == 0 for value in values)
            / number_of_queries,
            "full_recall_rate": sum(value == 1 for value in values)
            / number_of_queries,
        }

    return {
        "number_of_queries": number_of_queries,
        "candidate_recall": candidate_recall,
        "mrr": sum(reciprocal_ranks) / number_of_queries,
        "official_style_top_5": {
            "precision": sum(top_5_precision) / number_of_queries,
            "recall": sum(top_5_recall) / number_of_queries,
        },
        "queries_with_zero_recall_at_100": len(zero_recall_at_100_ids),
        "queries_with_full_recall_at_100": full_recall_at_100,
        "queries_where_gold_appears_only_after_rank_5": gold_only_after_rank_5,
        "zero_recall_at_100_sample_ids": zero_recall_at_100_ids[:20],
    }


def compare_official_style_top_5(
    samples: dict,
    reference_rankings: dict[str, list[str]],
    alternative_rankings: dict[str, list[str]],
) -> dict:
    """Count per-query top-5 precision and recall changes from a reference."""

    if not samples:
        raise ValueError("samples must not be empty")

    comparisons = {
        "precision": Counter(),
        "recall": Counter(),
    }
    for sample_id, sample in samples.items():
        gold_values = sample.get("answer")
        if not isinstance(gold_values, list) or not gold_values:
            raise ValueError(f"sample {sample_id!r}: expected a non-empty answer list")
        gold = {str(document_id) for document_id in gold_values}

        scores = {}
        for label, rankings in (
            ("reference", reference_rankings),
            ("alternative", alternative_rankings),
        ):
            top_5 = [
                str(document_id) for document_id in rankings.get(str(sample_id), [])[:5]
            ]
            if len(top_5) != len(set(top_5)):
                raise ValueError(
                    f"sample {sample_id!r}: {label} ranking contains duplicate IDs"
                )
            overlap = len(gold.intersection(top_5))
            scores[label] = {
                "precision": overlap / len(top_5) if top_5 else 0.0,
                "recall": overlap / len(gold),
            }

        for metric, metric_comparisons in comparisons.items():
            difference = scores["alternative"][metric] - scores["reference"][metric]
            label = (
                "improved"
                if difference > 0
                else "worsened"
                if difference < 0
                else "unchanged"
            )
            metric_comparisons[label] += 1

    return {
        metric: {
            label: comparisons[metric][label]
            for label in ("improved", "unchanged", "worsened")
        }
        for metric in comparisons
    }


def summarize_first_gold_ranks(samples: dict, rankings: dict[str, list[str]]) -> dict:
    """Summarize first-gold ranks over the complete candidate universe."""

    if not samples:
        raise ValueError("samples must not be empty")

    first_gold_ranks = []
    rank_bins = Counter()
    for sample_id, sample in samples.items():
        gold_values = sample.get("answer")
        if not isinstance(gold_values, list) or not gold_values:
            raise ValueError(f"sample {sample_id!r}: expected a non-empty answer list")
        gold = {str(document_id) for document_id in gold_values}
        ranked = [str(document_id) for document_id in rankings.get(str(sample_id), [])]
        if len(ranked) != len(set(ranked)):
            raise ValueError(f"sample {sample_id!r}: ranking contains duplicate IDs")

        first_gold_rank = next(
            (
                rank
                for rank, document_id in enumerate(ranked, start=1)
                if document_id in gold
            ),
            None,
        )
        if first_gold_rank is None:
            rank_bins["not_found"] += 1
            continue

        first_gold_ranks.append(first_gold_rank)
        if first_gold_rank == 1:
            rank_bins["1"] += 1
        elif first_gold_rank <= 5:
            rank_bins["2-5"] += 1
        elif first_gold_rank <= 10:
            rank_bins["6-10"] += 1
        elif first_gold_rank <= 20:
            rank_bins["11-20"] += 1
        elif first_gold_rank <= 50:
            rank_bins["21-50"] += 1
        elif first_gold_rank <= 100:
            rank_bins["51-100"] += 1
        elif first_gold_rank <= 200:
            rank_bins["101-200"] += 1
        else:
            rank_bins["beyond_200"] += 1

    ordered_bin_labels = (
        "1",
        "2-5",
        "6-10",
        "11-20",
        "21-50",
        "51-100",
        "101-200",
        "beyond_200",
        "not_found",
    )
    return {
        "when_found": {
            "median": float(median(first_gold_ranks)) if first_gold_ranks else None,
            "p90": _percentile(first_gold_ranks, 90),
            "p95": _percentile(first_gold_ranks, 95),
            "max": max(first_gold_ranks, default=None),
        },
        "counts": {label: rank_bins[label] for label in ordered_bin_labels},
    }


MODEL_NAME = "BAAI/bge-reranker-v2-m3"


MODEL_REVISION = "953dc6f"


MODEL_RESOLVED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"


TOP_K_CHUNKS = 2_000


CANDIDATE_DEPTH = 100


SUPPORTING_CHUNKS_PER_DOCUMENT = 2


MAX_SEQUENCE_LENGTH_CAP = 8_192


def load_reranker(
    model_path: str | Path,
    model_name: str = MODEL_NAME,
    revision: str = MODEL_REVISION,
    device: str | None = None,
) -> dict:
    """Load the fixed reranker only from an attached local snapshot."""

    selected_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if selected_device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is not available")
    model_dtype = torch.float16 if selected_device.startswith("cuda") else torch.float32
    resolved_model_path = Path(model_path).expanduser().resolve()
    if not resolved_model_path.is_dir():
        raise FileNotFoundError(
            f"complete local model snapshot directory does not exist: {resolved_model_path}"
        )
    model_source = str(resolved_model_path)

    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        model_source,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_source,
        dtype=model_dtype,
        local_files_only=True,
    )
    model.to(selected_device)
    model.eval()

    resolved_revision = resolved_model_path.name
    if resolved_revision != MODEL_RESOLVED_REVISION:
        raise RuntimeError(
            f"model resolved to unexpected revision {resolved_revision!r}"
        )

    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    max_sequence_length = min(
        tokenizer_limit,
        model_limit,
        MAX_SEQUENCE_LENGTH_CAP,
    )
    if max_sequence_length <= 0:
        raise ValueError("resolved maximum sequence length must be positive")

    return {
        "model": model,
        "tokenizer": tokenizer,
        "model_name": model_name,
        "requested_revision": revision,
        "resolved_revision": resolved_revision,
        "device": selected_device,
        "dtype": str(model_dtype).removeprefix("torch."),
        "max_sequence_length": max_sequence_length,
        "load_seconds": perf_counter() - started,
    }


def build_fixed_candidates(
    index: dict,
    chunks: list[dict],
    samples: dict,
    top_k_chunks: int = TOP_K_CHUNKS,
    candidate_depth: int = CANDIDATE_DEPTH,
    batch_size: int = 64,
) -> dict:
    """Build top-100 sum-top-2 documents with their best two BM25 chunks."""

    if top_k_chunks <= 0:
        raise ValueError("top_k_chunks must be greater than zero")
    if candidate_depth <= 0:
        raise ValueError("candidate_depth must be greater than zero")
    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero")
    if top_k_chunks > index["number_of_chunks"]:
        raise ValueError("top_k_chunks cannot exceed the number of indexed chunks")
    if len(chunks) != index["number_of_chunks"]:
        raise ValueError("chunks must be the exact sequence used to build the index")

    sample_items = list(samples.items())
    candidates_by_query = {}
    started = perf_counter()

    for batch_start in range(0, len(sample_items), batch_size):
        batch = sample_items[batch_start : batch_start + batch_size]
        query_tokens = []
        for sample_id, sample in batch:
            question = sample.get("question")
            if not isinstance(question, str):
                raise TypeError(f"sample {sample_id!r}: question must be a string")
            query_tokens.append(lexical_tokenize(question))

        retrieval_result = index["retriever"].retrieve(
            query_tokens,
            k=top_k_chunks,
            sorted=True,
            return_as="tuple",
            show_progress=False,
        )

        for (sample_id, _), hit_indices, hit_scores in zip(
            batch, retrieval_result.documents, retrieval_result.scores
        ):
            chunk_hits = []
            for chunk_rank, (chunk_index_value, score_value) in enumerate(
                zip(hit_indices, hit_scores), start=1
            ):
                chunk_index = int(chunk_index_value)
                chunk_id, document_id = index["chunk_metadata"][chunk_index]
                chunk = chunks[chunk_index]
                if (
                    str(chunk["chunk_id"]) != chunk_id
                    or str(chunk["document_id"]) != document_id
                ):
                    raise RuntimeError("chunk order does not match the BM25 index")
                chunk_hits.append(
                    {
                        "chunk_id": chunk_id,
                        "document_id": document_id,
                        "score": float(score_value),
                        "chunk_rank": chunk_rank,
                        "text": chunk["text"],
                    }
                )

            aggregated = aggregate_documents(chunk_hits, method="sum_top_2")
            selected_documents = aggregated[:candidate_depth]
            if len(selected_documents) != candidate_depth:
                raise RuntimeError(
                    f"sample {sample_id!r}: expected {candidate_depth} candidates, "
                    f"got {len(selected_documents)}"
                )

            by_document = {document["document_id"]: [] for document in selected_documents}
            for hit in chunk_hits:
                supporting_chunks = by_document.get(hit["document_id"])
                if (
                    supporting_chunks is not None
                    and len(supporting_chunks) < SUPPORTING_CHUNKS_PER_DOCUMENT
                ):
                    supporting_chunks.append(
                        {
                            "chunk_id": hit["chunk_id"],
                            "text": hit["text"],
                            "bm25_rank": hit["chunk_rank"],
                            "bm25_score": hit["score"],
                        }
                    )

            query_candidates = []
            for original_rank, document in enumerate(selected_documents, start=1):
                supporting_chunks = by_document[document["document_id"]]
                if not supporting_chunks:
                    raise RuntimeError("candidate document has no supporting chunk")
                query_candidates.append(
                    {
                        "document_id": document["document_id"],
                        "original_rank": original_rank,
                        "original_score": document["score"],
                        "supporting_chunks": supporting_chunks,
                    }
                )

            candidate_ids = [candidate["document_id"] for candidate in query_candidates]
            if len(candidate_ids) != len(set(candidate_ids)):
                raise RuntimeError(f"sample {sample_id!r}: duplicate candidate IDs")
            candidates_by_query[str(sample_id)] = query_candidates

    return {
        "candidates_by_query": candidates_by_query,
        "reference_rankings": {
            sample_id: [candidate["document_id"] for candidate in candidates]
            for sample_id, candidates in candidates_by_query.items()
        },
        "top_k_chunks": top_k_chunks,
        "candidate_depth": candidate_depth,
        "retrieval_seconds": perf_counter() - started,
    }


def score_query_chunk_pairs(
    reranker: dict,
    pairs: list[tuple[str, str]],
    batch_size: int = 8,
    progress: Callable[[int, int], None] | None = None,
) -> dict:
    """Score query/chunk pairs and report pre-truncation token diagnostics."""

    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero")
    if not pairs:
        raise ValueError("pairs must not be empty")

    tokenizer = reranker["tokenizer"]
    model = reranker["model"]
    device = reranker["device"]
    max_sequence_length = reranker["max_sequence_length"]
    scores = []
    token_lengths = []
    model_forward_seconds = 0.0
    scoring_started = perf_counter()

    for batch_start in range(0, len(pairs), batch_size):
        batch = pairs[batch_start : batch_start + batch_size]
        questions = [pair[0] for pair in batch]
        passages = [pair[1] for pair in batch]
        untruncated = tokenizer(
            questions,
            passages,
            padding=False,
            truncation=False,
            return_length=True,
        )
        token_lengths.extend(int(length) for length in untruncated["length"])

        inputs = tokenizer(
            questions,
            passages,
            padding=True,
            truncation="only_second",
            max_length=max_sequence_length,
            return_tensors="pt",
        )
        inputs = {name: value.to(device) for name, value in inputs.items()}

        if device.startswith("cuda"):
            torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = model(**inputs, return_dict=True).logits.view(-1).float()
        if device.startswith("cuda"):
            torch.cuda.synchronize()
        model_forward_seconds += perf_counter() - forward_started

        batch_scores = logits.cpu().tolist()
        if len(batch_scores) != len(batch) or not all(
            isfinite(score) for score in batch_scores
        ):
            raise RuntimeError("reranker returned invalid scores")
        scores.extend(float(score) for score in batch_scores)

        completed = min(batch_start + len(batch), len(pairs))
        if progress is not None:
            progress(completed, len(pairs))

    lengths = np.asarray(token_lengths, dtype=np.int32)
    truncated = int(np.sum(lengths > max_sequence_length))
    scoring_seconds = perf_counter() - scoring_started
    return {
        "scores": scores,
        "diagnostics": {
            "number_of_pairs": len(pairs),
            "token_length_before_truncation": {
                "median": float(np.median(lengths)),
                "p95": float(np.percentile(lengths, 95)),
                "max": int(lengths.max()),
            },
            "truncated_pairs": truncated,
            "truncated_fraction": truncated / len(pairs),
            "model_forward_seconds": model_forward_seconds,
            "scoring_seconds": scoring_seconds,
            "model_forward_pairs_per_second": len(pairs) / model_forward_seconds,
            "end_to_end_pairs_per_second": len(pairs) / scoring_seconds,
        },
    }


def rerank_documents(
    candidates: list[dict], supporting_chunk_scores: list[list[float]]
) -> list[dict]:
    """Sum up to two chunk scores and deterministically rerank all candidates."""

    if len(candidates) != len(supporting_chunk_scores):
        raise ValueError("each candidate must have one supporting-score list")

    candidate_ids = [str(candidate["document_id"]) for candidate in candidates]
    if len(candidate_ids) != len(set(candidate_ids)):
        raise ValueError("candidate document IDs must be unique")

    reranked = []
    for candidate, scores in zip(candidates, supporting_chunk_scores):
        expected_scores = len(candidate["supporting_chunks"])
        if not 1 <= expected_scores <= SUPPORTING_CHUNKS_PER_DOCUMENT:
            raise ValueError("candidate must have one or two supporting chunks")
        if len(scores) != expected_scores:
            raise ValueError("supporting-score count does not match supporting chunks")
        canonical_scores = [float(score) for score in scores]
        if not all(isfinite(score) for score in canonical_scores):
            raise ValueError("cross-encoder scores must be finite")
        reranked.append(
            {
                **candidate,
                "cross_encoder_chunk_scores": canonical_scores,
                "cross_encoder_score": sum(canonical_scores),
            }
        )

    reranked.sort(
        key=lambda document: (
            -document["cross_encoder_score"],
            document["original_rank"],
            str(document["document_id"]),
        )
    )
    if {document["document_id"] for document in reranked} != set(candidate_ids):
        raise RuntimeError("reranking changed the candidate set")
    return reranked


def rerank_fixed_candidates(
    reranker: dict,
    samples: dict,
    candidates_by_query: dict[str, list[dict]],
    batch_size: int = 8,
    progress: Callable[[int, int], None] | None = None,
) -> dict:
    """Score independent query/chunk pairs and rerank each fixed candidate set."""

    pairs = []
    score_counts = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        if not isinstance(question, str):
            raise TypeError(f"sample {sample_id!r}: question must be a string")
        candidates = candidates_by_query.get(str(sample_id))
        if candidates is None:
            raise KeyError(f"sample {sample_id!r}: missing fixed candidate set")
        query_counts = []
        for candidate in candidates:
            support_count = len(candidate["supporting_chunks"])
            query_counts.append(support_count)
            pairs.extend(
                (question, supporting_chunk["text"])
                for supporting_chunk in candidate["supporting_chunks"]
            )
        score_counts.append((str(sample_id), query_counts))

    scoring = score_query_chunk_pairs(
        reranker,
        pairs,
        batch_size=batch_size,
        progress=progress,
    )
    score_offset = 0
    reranked_by_query = {}
    for sample_id, query_counts in score_counts:
        document_scores = []
        for support_count in query_counts:
            document_scores.append(
                scoring["scores"][score_offset : score_offset + support_count]
            )
            score_offset += support_count
        reranked_by_query[sample_id] = rerank_documents(
            candidates_by_query[sample_id], document_scores
        )
    if score_offset != len(scoring["scores"]):
        raise RuntimeError("not every cross-encoder score was consumed")

    return {
        "reranked_by_query": reranked_by_query,
        "rankings": {
            sample_id: [document["document_id"] for document in documents]
            for sample_id, documents in reranked_by_query.items()
        },
        "diagnostics": scoring["diagnostics"],
    }


def paired_bootstrap(
    reference: list[float],
    alternative: list[float],
    seed: int = 2_026_091_3,
    resamples: int = 10_000,
) -> dict:
    """Bootstrap a paired mean delta over per-query metric contributions."""

    reference_values = np.asarray(reference, dtype=np.float64)
    alternative_values = np.asarray(alternative, dtype=np.float64)
    if reference_values.shape != alternative_values.shape or reference_values.ndim != 1:
        raise ValueError("reference and alternative must be same-length vectors")
    if reference_values.size == 0:
        raise ValueError("metric contributions must not be empty")
    if resamples <= 0:
        raise ValueError("resamples must be greater than zero")

    paired_deltas = alternative_values - reference_values
    generator = np.random.default_rng(seed)
    bootstrap_deltas = np.empty(resamples, dtype=np.float64)
    for resample_index in range(resamples):
        indices = generator.integers(0, paired_deltas.size, paired_deltas.size)
        bootstrap_deltas[resample_index] = paired_deltas[indices].mean()

    return {
        "observed_delta": float(paired_deltas.mean()),
        "bootstrap_mean": float(bootstrap_deltas.mean()),
        "percentile_interval_95": [
            float(np.percentile(bootstrap_deltas, 2.5)),
            float(np.percentile(bootstrap_deltas, 97.5)),
        ],
        "fraction_delta_greater_than_zero": float(np.mean(bootstrap_deltas > 0)),
        "seed": seed,
        "resamples": resamples,
    }



def official_eval_retrieval(predictions: dict, truth: dict) -> dict:
    predicted = {sample_id: value['answer'] for sample_id, value in predictions.items()}
    predicted_ids = list(predicted)
    truth_ids = list(truth)
    if len(predicted_ids) != len(truth_ids):
        raise ValueError('Samples in predictions do not match the reference')

    recall = np.array([
        len(set(truth[sample_id]) & set(predicted.get(sample_id, set())))
        / len(truth[sample_id])
        if 0 < len(predicted.get(sample_id, [])) <= 5 else 0
        for sample_id in truth_ids
    ]).mean()
    precision = np.array([
        len(set(truth[sample_id]) & set(predicted.get(sample_id, set())))
        / len(predicted[sample_id])
        if 0 < len(predicted.get(sample_id, [])) <= 5 else 0
        for sample_id in predicted_ids
    ]).mean()
    return {'precision': float(precision), 'recall': float(recall)}


def official_contributions(predictions: dict, truth: dict) -> dict:
    contributions = {'precision': [], 'recall': []}
    for sample_id, answer in truth.items():
        scores = official_eval_retrieval(
            {sample_id: predictions[sample_id]},
            {sample_id: answer},
        )
        for metric in contributions:
            contributions[metric].append(scores[metric])
    return contributions


def prepare_dev_samples(source_path: Path) -> tuple[dict, dict]:
    samples = load_legal_ir(source_path)
    split_ids = make_legalir_split(samples)
    dev_samples = select_samples(samples, split_ids['dev'])
    return dev_samples, {
        'version': 'legalir_split_v1',
        'method': 'sha256(exact raw question UTF-8), first 8 hex digits modulo 100',
        'source_sha256': sha256(source_path.read_bytes()).hexdigest(),
        'source_queries': len(samples),
        'dev_queries': len(dev_samples),
    }


def run_dev_experiment(
    legalir_source_path: str | Path,
    corpus_path: str | Path,
    model_path: str | Path,
    batch_size: int = 128,
    smoke_queries: int = 2,
) -> dict:
    dev_samples, split_info = prepare_dev_samples(Path(legalir_source_path))
    documents = load_corpus(Path(corpus_path))
    chunks = chunk_corpus(documents, chunk_size=2_000, overlap=200)
    index = build_bm25(chunks, k1=1.5, b=0.75)
    reranker = load_reranker(model_path=model_path, device='cuda')

    smoke_samples = dict(list(dev_samples.items())[:smoke_queries])
    smoke_candidates = build_fixed_candidates(index, chunks, smoke_samples)
    smoke_result = rerank_fixed_candidates(
        reranker,
        smoke_samples,
        smoke_candidates['candidates_by_query'],
        batch_size=batch_size,
    )
    smoke_sets_preserved = all(
        set(smoke_candidates['reference_rankings'][sample_id])
        == set(smoke_result['rankings'][sample_id])
        for sample_id in smoke_samples
    )
    deterministic_smoke = True
    for sample_id in smoke_samples:
        scored_by_id = {
            document['document_id']: document
            for document in smoke_result['reranked_by_query'][sample_id]
        }
        original_candidates = smoke_candidates['candidates_by_query'][sample_id]
        score_lists = [
            scored_by_id[candidate['document_id']]['cross_encoder_chunk_scores']
            for candidate in original_candidates
        ]
        repeated = rerank_documents(original_candidates, score_lists)
        deterministic_smoke &= [
            item['document_id'] for item in repeated
        ] == smoke_result['rankings'][sample_id]

    fixed_candidates = build_fixed_candidates(index, chunks, dev_samples)
    torch.cuda.reset_peak_memory_stats()
    last_progress_time = [perf_counter()]

    def show_progress(completed: int, total: int) -> None:
        now = perf_counter()
        if completed == total or now - last_progress_time[0] >= 30:
            print(f'reranking pairs: {completed}/{total}', flush=True)
            last_progress_time[0] = now

    reranked = rerank_fixed_candidates(
        reranker,
        dev_samples,
        fixed_candidates['candidates_by_query'],
        batch_size=batch_size,
        progress=show_progress,
    )
    reference_rankings = fixed_candidates['reference_rankings']
    alternative_rankings = reranked['rankings']
    candidate_sets_identical = sum(
        set(reference_rankings[sample_id]) == set(alternative_rankings[sample_id])
        for sample_id in dev_samples
    )

    depths = (5, 10, 20, 50, 100)
    reference_internal = evaluate_retrieval(
        dev_samples, reference_rankings, candidate_depths=depths
    )
    alternative_internal = evaluate_retrieval(
        dev_samples, alternative_rankings, candidate_depths=depths
    )
    reference_ceiling = reference_internal['candidate_recall'][100]

    truth = {
        sample_id: sample['answer'] for sample_id, sample in dev_samples.items()
    }
    reference_predictions = make_legalir_predictions(reference_rankings)
    alternative_predictions = make_legalir_predictions(alternative_rankings)
    reference_official = official_eval_retrieval(reference_predictions, truth)
    alternative_official = official_eval_retrieval(alternative_predictions, truth)

    deltas = {
        'bundled_precision': (
            alternative_official['precision'] - reference_official['precision']
        ),
        'bundled_recall': alternative_official['recall'] - reference_official['recall'],
        'mrr': alternative_internal['mrr'] - reference_internal['mrr'],
    }
    for depth in (10, 20, 50):
        deltas[f'recall_at_{depth}'] = (
            alternative_internal['candidate_recall'][depth]['mean']
            - reference_internal['candidate_recall'][depth]['mean']
        )

    bootstrap = None
    if deltas['bundled_precision'] > 0 and deltas['bundled_recall'] > 0:
        reference_contributions = official_contributions(reference_predictions, truth)
        alternative_contributions = official_contributions(alternative_predictions, truth)
        bootstrap = {
            metric: paired_bootstrap(
                reference_contributions[metric], alternative_contributions[metric]
            )
            for metric in ('precision', 'recall')
        }

    runtime = {
        'model_load_seconds': reranker['load_seconds'],
        'candidate_retrieval_seconds': fixed_candidates['retrieval_seconds'],
        **reranked['diagnostics'],
        'peak_gpu_memory_bytes': int(torch.cuda.max_memory_allocated()),
    }
    return {
        'split': split_info,
        'controls': {
            'documents': len(documents),
            'chunks': len(chunks),
            'chunk_size': 2_000,
            'overlap': 200,
            'bm25_library': f'bm25s=={bm25s.__version__}',
            'bm25_method': 'lucene',
            'k1': 1.5,
            'b': 0.75,
            'top_k_chunks': TOP_K_CHUNKS,
            'aggregation': 'sum top-2',
            'candidate_depth': CANDIDATE_DEPTH,
        },
        'reranker': {
            'model': reranker['model_name'],
            'requested_revision': reranker['requested_revision'],
            'resolved_revision': reranker['resolved_revision'],
            'tokenizer': reranker['tokenizer'].name_or_path,
            'max_sequence_length': reranker['max_sequence_length'],
            'torch_version': torch.__version__,
            'transformers_version': transformers.__version__,
            'device': reranker['device'],
            'dtype': reranker['dtype'],
            'batch_size': batch_size,
        },
        'smoke_test': {
            'queries': smoke_queries,
            'score_shape': [smoke_result['diagnostics']['number_of_pairs']],
            'finite_scores': True,
            'candidate_sets_preserved': smoke_sets_preserved,
            'deterministic_ordering': deterministic_smoke,
        },
        'candidate_sets_identical': {
            'queries': candidate_sets_identical,
            'total_queries': len(dev_samples),
        },
        'candidate_recall_at_100_ceiling': reference_ceiling,
        'reference': {
            'bundled_scorer': reference_official,
            'internal': reference_internal,
            'first_gold_rank': summarize_first_gold_ranks(
                dev_samples, reference_rankings
            ),
        },
        'reranked': {
            'bundled_scorer': alternative_official,
            'internal': alternative_internal,
            'first_gold_rank': summarize_first_gold_ranks(
                dev_samples, alternative_rankings
            ),
        },
        'deltas': deltas,
        'paired_top_5': compare_official_style_top_5(
            dev_samples, reference_rankings, alternative_rankings
        ),
        'paired_bootstrap': bootstrap,
        'runtime': runtime,
    }


In [ ]:
BATCH_SIZE = 128
result = run_dev_experiment(
    legalir_source_path=LEGALIR_SOURCE_PATH,
    corpus_path=CORPUS_PATH,
    model_path=MODEL_PATH,
    batch_size=BATCH_SIZE,
    smoke_queries=2,
)

assert result['smoke_test']['finite_scores']
assert result['smoke_test']['candidate_sets_preserved']
assert result['smoke_test']['deterministic_ordering']
assert (
    result['candidate_sets_identical']['queries']
    == result['candidate_sets_identical']['total_queries']
)

output_path = Path('/kaggle/working/zero_shot_cross_encoder_reranking_dev_results.json')
output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', output_path)


In [ ]:
summary = {
    'smoke_test': result['smoke_test'],
    'candidate_sets_identical': result['candidate_sets_identical'],
    'candidate_recall_at_100_ceiling': result['candidate_recall_at_100_ceiling'],
    'reference_bundled_scorer': result['reference']['bundled_scorer'],
    'reranked_bundled_scorer': result['reranked']['bundled_scorer'],
    'deltas': result['deltas'],
    'paired_top_5': result['paired_top_5'],
    'paired_bootstrap': result['paired_bootstrap'],
    'runtime': result['runtime'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
